In [8]:

# Hằng số tỷ lệ lấy top tần số
TOP_K_RATE = 0.5

In [9]:
from PIL import Image
import os

# === Config ===
image_path = "private_test/X_test/"  # Đường dẫn đến ảnh gốc
ROWS = 3
COLS = 5

def split_image(image_path, rows, cols):
    """Đọc ảnh và cắt thành rows x cols, trả về danh sách các mảnh."""
    img = Image.open(image_path)
    width, height = img.size

    piece_width = width // cols
    piece_height = height // rows

    pieces = []
    for r in range(rows):
        for c in range(cols):
            left = c * piece_width
            upper = r * piece_height
            right = left + piece_width
            lower = upper + piece_height
            piece = img.crop((left, upper, right, lower))
            pieces.append(piece)

    return pieces

def load_all_pieces(image_dir, rows, cols):
    """
    Duyệt toàn bộ ảnh trong thư mục và trả về mảng 3 chiều:
    all_pieces[image_index][row][col] = PIL.Image
    """
    all_pieces = []
    image_files = sorted(
        [f for f in os.listdir(image_dir) if f.lower().endswith(('.png', '.jpg', '.jpeg'))]
    )

    print(f"Found {len(image_files)} images in {image_dir}")

    for filename in image_files:
        image_path = os.path.join(image_dir, filename)
        print(f"- Processing {filename}...")
        pieces = split_image(image_path, rows, cols)
        all_pieces.append(pieces)

    print(f"\nTotal images processed: {len(all_pieces)}")
    return all_pieces

all_pieces = load_all_pieces(image_path, ROWS, COLS)
# if DEBUG:
#     all_pieces = all_pieces[:20]  # Giới hạn để debug nhanh

# Ví dụ: xem thông tin
print(f"Total images: {len(all_pieces)}")  # số lượng ảnh
if all_pieces:
    print(f"Rows per image: {len(all_pieces[0])}")


Found 100 images in private_test/X_test/
- Processing Alfred_Sisley_115_shuffled.jpg...
- Processing Alfred_Sisley_188_shuffled.jpg...
- Processing Alfred_Sisley_205_shuffled.jpg...
- Processing Alfred_Sisley_232_shuffled.jpg...
- Processing Alfred_Sisley_6_shuffled.jpg...
- Processing Amedeo_Modigliani_112_shuffled.jpg...
- Processing Amedeo_Modigliani_143_shuffled.jpg...
- Processing Amedeo_Modigliani_2_shuffled.jpg...
- Processing Amedeo_Modigliani_65_shuffled.jpg...
- Processing Amedeo_Modigliani_73_shuffled.jpg...
- Processing Andrei_Rublev_32_shuffled.jpg...
- Processing Andrei_Rublev_50_shuffled.jpg...
- Processing Andrei_Rublev_89_shuffled.jpg...
- Processing Andy_Warhol_11_shuffled.jpg...
- Processing Andy_Warhol_161_shuffled.jpg...
- Processing Andy_Warhol_171_shuffled.jpg...
- Processing Andy_Warhol_70_shuffled.jpg...
- Processing Andy_Warhol_88_shuffled.jpg...
- Processing Camille_Pissarro_3_shuffled.jpg...
- Processing Caravaggio_19_shuffled.jpg...
- Processing Caravaggio_

In [10]:
import numpy as np
from PIL import Image
import os
import csv
import random
from tqdm import tqdm
import pandas as pd

# =========================
# HELPER FUNCTIONS
# =========================
def _to_np_rgb(pil_img):
    return np.asarray(pil_img.convert("RGB"), dtype=np.float32)

def _edge_signal_gray(edge_rgb):
    if edge_rgb.ndim == 1:
        return edge_rgb.astype(np.float32)
    r = edge_rgb[:, 0]
    g = edge_rgb[:, 1]
    b = edge_rgb[:, 2]
    return 0.299 * r + 0.587 * g + 0.114 * b

def _edge_signature(edge_rgb, top_k_rate=TOP_K_RATE, phase_bins=12):
    phase_bins = max(1, int(phase_bins))
    gray = _edge_signal_gray(edge_rgb)
    
    if gray.size < 2 or top_k_rate <= 0:
        return frozenset()
        
    gray = gray.astype(np.float32)
    gray = gray - float(np.mean(gray))
    fft = np.fft.rfft(gray)
    mags = np.abs(fft)
    
    if mags.size <= 1:
        return frozenset()
        
    # Tính toán k động dựa trên top_k_rate (loại bỏ thành phần DC ở index 0)
    k = max(1, int((mags.size - 1) * top_k_rate))
    k = min(k, mags.size - 1)
    
    idx = np.arange(1, mags.size)
    top = idx[np.argpartition(mags[idx], -k)[-k:]]
    phases = np.angle(fft[top])
    phase_norm = (phases + (2.0 * np.pi)) % (2.0 * np.pi)
    bins = np.floor(phase_norm / (2.0 * np.pi) * phase_bins).astype(int)
    return frozenset((int(freq), int(bin_)) for freq, bin_ in zip(top, bins))

def _extract_raw_borders(pieces):
    """Trích xuất mảng pixel thô của 4 cạnh (chưa qua xử lý)"""
    arrs = [_to_np_rgb(p) for p in pieces]
    tops    = [a[0, :, :]  for a in arrs]  # (W,3)
    bottoms = [a[-1, :, :] for a in arrs]  # (W,3)
    lefts   = [a[:, 0, :]  for a in arrs]  # (H,3)
    rights  = [a[:, -1, :] for a in arrs]  # (H,3)
    return tops, bottoms, lefts, rights

def _signature_cost(sig_a, sig_b):
    if not sig_a and not sig_b:
        return 0.0
    inter = len(sig_a & sig_b)
    union = len(sig_a | sig_b)
    return 1.0 - inter / union if union else 0.0

# =========================
# HYBRID COST: Fourier + MSE
# =========================
def compute_cost_matrix_hybrid(pieces, top_k_rate=TOP_K_RATE, phase_bins=12, alpha=0.5):
    """
    Tính ma trận cost kết hợp giữa Fourier Signature và Pixel MSE.
    
    Tham số:
    - alpha (float): Trọng số của Fourier. 
                     alpha = 1.0 -> Chỉ dùng Fourier (giống code cũ).
                     alpha = 0.0 -> Chỉ dùng MSE pixel.
                     alpha = 0.5 -> Cân bằng 50/50.
                     
    - H[i,j]: cost đặt j bên phải i  -> so sánh right(i) vs left(j)
    - V[i,j]: cost đặt j bên dưới i  -> so sánh bottom(i) vs top(j)
    """
    n = len(pieces)
    H = np.zeros((n, n), dtype=np.float64)
    V = np.zeros((n, n), dtype=np.float64)

    # 1. Trích xuất raw border pixels
    tops, bottoms, lefts, rights = _extract_raw_borders(pieces)

    # 2. Tính toán các Fourier signatures từ các cạnh thô
    tops_s    = [_edge_signature(t, top_k_rate, phase_bins) for t in tops]
    bottoms_s = [_edge_signature(b, top_k_rate, phase_bins) for b in bottoms]
    lefts_s   = [_edge_signature(l, top_k_rate, phase_bins) for l in lefts]
    rights_s  = [_edge_signature(r, top_k_rate, phase_bins) for r in rights]

    # 3. Tính toán ma trận Cost kết hợp
    for i in range(n):
        # Thông tin chữ ký của mảnh i
        r_i_sig = rights_s[i]
        b_i_sig = bottoms_s[i]
        
        # Pixel của mảnh i (chuẩn hóa / 255.0 để đưa MSE về dải hẹp tương đương Jaccard)
        r_i_px = rights[i] / 255.0
        b_i_px = bottoms[i] / 255.0

        for j in range(n):
            if i == j:
                continue
                
            # Chuẩn hóa pixel của mảnh j
            l_j_px = lefts[j] / 255.0
            t_j_px = tops[j] / 255.0

            # ----- HORIZONTAL COST (Right i ghép với Left j) -----
            sig_cost_h = _signature_cost(r_i_sig, lefts_s[j])
            mse_cost_h = np.mean((r_i_px - l_j_px) ** 2) 
            H[i, j] = alpha * sig_cost_h + (1.0 - alpha) * mse_cost_h

            # ----- VERTICAL COST (Bottom i ghép với Top j) -----
            sig_cost_v = _signature_cost(b_i_sig, tops_s[j])
            mse_cost_v = np.mean((b_i_px - t_j_px) ** 2)
            V[i, j] = alpha * sig_cost_v + (1.0 - alpha) * mse_cost_v

    return H, V

# Alias để tương thích ngược với các file khác trong notebook của bạn
compute_cost_matrix_mse = compute_cost_matrix_hybrid
compute_cost_matrix_signature = compute_cost_matrix_hybrid

In [11]:
# =========================
# LOCAL SEARCH (first improvement)
# Chromosome = c->o (cell -> piece)
# =========================

class LocalSearchPuzzleSolver:
    def __init__(self, rows, cols, max_iters=1000, restarts=5, rng_seed=None):
        self.rows = rows
        self.cols = cols
        self.max_iters = max_iters
        self.restarts = restarts
        self.rng = random.Random(rng_seed)

    # ------ utils ------
    def _repair_perm(self, chrom):
        """Đảm bảo chrom là hoán vị 0..N-1 (loại trùng/ngoài miền, chèn thiếu)."""
        n = self.rows * self.cols
        used, out = set(), []

        for g in chrom:
            if isinstance(g, (int, np.integer)) and 0 <= g < n and g not in used:
                out.append(int(g))
                used.add(int(g))

        out.extend([x for x in range(n) if x not in used])
        return out[:n]

    def _fitness(self, chrom, H, V):
        total = 0.0
        grid = np.array(chrom, dtype=int).reshape(self.rows, self.cols)

        for r in range(self.rows):
            for c in range(self.cols):
                cur = grid[r, c]

                if c < self.cols - 1:
                    total += H[cur, grid[r, c + 1]]

                if r < self.rows - 1:
                    total += V[cur, grid[r + 1, c]]

        return float(total)

    def _random_chrom(self, n):
        base = list(range(n))
        self.rng.shuffle(base)
        return base

    # ------ láng giềng: SWAP & BLOCK-SWAP (first improvement) ------
    def _first_improvement_swap_and_blockswap(self, chrom, H, V, base_fit):
        R, C = self.rows, self.cols
        n = len(chrom)

        # 1) Tất cả SWAP i < j
        # First improvement: gặp cải thiện là return ngay
        for i in range(n - 1):
            for j in range(i + 1, n):
                cand = chrom[:]
                cand[i], cand[j] = cand[j], cand[i]
                cand = self._repair_perm(cand)

                d = self._fitness(cand, H, V) - base_fit

                if d < 0.0:
                    return cand, d

        # 2) Tất cả BLOCK-SWAP
        # Duyệt mọi h, w; mọi cặp vị trí; cho phép overlap như code gốc
        grid0 = np.array(chrom, dtype=int).reshape(R, C)

        for h in range(1, R + 1):
            for w in range(1, C + 1):
                positions = [
                    (r, c)
                    for r in range(0, R - h + 1)
                    for c in range(0, C - w + 1)
                ]

                m = len(positions)
                if m <= 1:
                    continue

                for a in range(m - 1):
                    r0, c0 = positions[a]

                    for b in range(a + 1, m):
                        r1, c1 = positions[b]

                        if r0 == r1 and c0 == c1:
                            continue

                        g = grid0.copy()

                        src = g[r0:r0 + h, c0:c0 + w].copy()
                        dst = g[r1:r1 + h, c1:c1 + w].copy()

                        g[r0:r0 + h, c0:c0 + w] = dst
                        g[r1:r1 + h, c1:c1 + w] = src

                        cand = self._repair_perm(g.reshape(-1).tolist())

                        d = self._fitness(cand, H, V) - base_fit

                        if d < 0.0:
                            return cand, d

        return None, 0.0

    # ------ láng giềng: “1-đổi-2” dọc & ngang (first improvement) ------
    def _first_improvement_one_with_two(self, chrom, H, V, base_fit):
        """
        Dọc:
            chọn dải cột (w, c0), phủ toàn bộ R;
            cắt theo chiều dọc tại r1 < r2:
            [Top=A, Mid=B, Bot=C] -> [C, A, B]

        Ngang:
            chọn dải hàng (h, r0), phủ toàn bộ C;
            cắt theo chiều ngang tại c1 < c2:
            [Left=A, Mid=B, Right=C] -> [C, A, B]
        """
        R, C = self.rows, self.cols
        grid0 = np.array(chrom, dtype=int).reshape(R, C)

        # --- 1) Dọc ---
        # r1, r2 là đường cắt giữa các hàng => trong [1..R-1]
        if R >= 3:
            cut_rows = [i for i in range(1, R)]

            for w in range(1, C + 1):
                for c0 in range(0, C - w + 1):
                    for a in range(len(cut_rows) - 1):
                        r1 = cut_rows[a]

                        for b in range(a + 1, len(cut_rows)):
                            r2 = cut_rows[b]

                            g = grid0.copy()

                            A = g[0:r1, c0:c0 + w].copy()       # top
                            B = g[r1:r2, c0:c0 + w].copy()      # mid
                            Cbot = g[r2:R, c0:c0 + w].copy()    # bottom

                            # Xếp lại: [C, A, B]
                            len_C = len(Cbot)
                            len_A = len(A)
                            len_B = len(B)

                            g[0:len_C, c0:c0 + w] = Cbot
                            g[len_C:len_C + len_A, c0:c0 + w] = A
                            g[len_C + len_A:len_C + len_A + len_B, c0:c0 + w] = B

                            cand = self._repair_perm(g.reshape(-1).tolist())

                            d = self._fitness(cand, H, V) - base_fit

                            if d < 0.0:
                                return cand, d

        # --- 2) Ngang ---
        # c1, c2 là đường cắt giữa các cột => trong [1..C-1]
        if C >= 3:
            cut_cols = [j for j in range(1, C)]

            for h in range(1, R + 1):
                for r0 in range(0, R - h + 1):
                    for a in range(len(cut_cols) - 1):
                        c1 = cut_cols[a]

                        for b in range(a + 1, len(cut_cols)):
                            c2 = cut_cols[b]

                            g = grid0.copy()

                            A = g[r0:r0 + h, 0:c1].copy()       # left
                            B = g[r0:r0 + h, c1:c2].copy()      # mid
                            Cright = g[r0:r0 + h, c2:C].copy()  # right

                            # Xếp lại: [C, A, B]
                            wC = Cright.shape[1]
                            wA = A.shape[1]
                            wB = B.shape[1]

                            g[r0:r0 + h, 0:wC] = Cright
                            g[r0:r0 + h, wC:wC + wA] = A
                            g[r0:r0 + h, wC + wA:wC + wA + wB] = B

                            cand = self._repair_perm(g.reshape(-1).tolist())

                            d = self._fitness(cand, H, V) - base_fit

                            if d < 0.0:
                                return cand, d

        return None, 0.0

    # ------ một bước first-improvement: gộp cả 3 nhóm láng giềng ------
    def _first_improvement_step(self, chrom, H, V):
        base_fit = self._fitness(chrom, H, V)

        # 1) swap + block-swap
        nb, d = self._first_improvement_swap_and_blockswap(chrom, H, V, base_fit)

        if nb is not None and d < 0.0:
            return nb, d

        # 2) one-with-two dọc + ngang
        nb, d = self._first_improvement_one_with_two(chrom, H, V, base_fit)

        if nb is not None and d < 0.0:
            return nb, d

        return None, 0.0

    # ------ main ------
    def run(self, pieces):
        """Trả về (best_order, best_fitness). Chromosome là c->o."""
        n = self.rows * self.cols
        H, V = compute_cost_matrix_mse(pieces)

        global_best, global_best_fit = None, float("inf")

        for _ in range(self.restarts):
            cur = self._repair_perm(self._random_chrom(n))
            cur_fit = self._fitness(cur, H, V)

            for _ in range(self.max_iters):
                nb, delta = self._first_improvement_step(cur, H, V)

                if nb is None or delta >= 0.0:
                    break

                cur = nb
                cur_fit += delta

            if cur_fit < global_best_fit:
                global_best_fit, global_best = cur_fit, cur

        return global_best, float(global_best_fit)

    def assemble_image(self, pieces, order):
        n = self.rows * self.cols

        if len(order) != n or set(order) != set(range(n)):
            order = self._repair_perm(order)

        grid = np.array(order, dtype=int).reshape(self.rows, self.cols)

        w, h = pieces[0].size
        out = Image.new("RGB", (w * self.cols, h * self.rows))

        for r in range(self.rows):
            for c in range(self.cols):
                out.paste(pieces[grid[r, c]], (c * w, r * h))

        return out


# =========================
# RUNNER + Evaluator (auto nhận biết biểu diễn)
# =========================

def invert_perm_c2o_to_o2c(c2o):
    """c2o[c] = o  ->  o2c[o] = c"""
    n = len(c2o)
    o2c = [0] * n

    for c, o in enumerate(c2o):
        o2c[o] = c

    return o2c


def invert_perm_o2c_to_c2o(o2c):
    """o2c[o] = c  ->  c2o[c] = o"""
    n = len(o2c)
    c2o = [0] * n

    for o, c in enumerate(o2c):
        c2o[c] = o

    return c2o


def _grid_from_c2o(c2o, rows, cols):
    return np.array(c2o, dtype=int).reshape(rows, cols)


def _adjacency_edges(grid):
    edges = set()
    R, C = grid.shape

    for r in range(R):
        for c in range(C):
            cur = int(grid[r, c])

            if c < C - 1:
                nxt = int(grid[r, c + 1])
                if cur != nxt:
                    edges.add((min(cur, nxt), max(cur, nxt)))

            if r < R - 1:
                nxt = int(grid[r + 1, c])
                if cur != nxt:
                    edges.add((min(cur, nxt), max(cur, nxt)))

    return edges


def _neighbor_accuracy_c2o(pred_c2o, gt_c2o, rows, cols):
    pred_edges = _adjacency_edges(_grid_from_c2o(pred_c2o, rows, cols))
    gt_edges = _adjacency_edges(_grid_from_c2o(gt_c2o, rows, cols))

    if not gt_edges:
        return 0.0

    return float(len(pred_edges & gt_edges) / len(gt_edges))


def _largest_component_accuracy_c2o(pred_c2o, gt_c2o, rows, cols):
    common_edges = (
        _adjacency_edges(_grid_from_c2o(pred_c2o, rows, cols))
        &
        _adjacency_edges(_grid_from_c2o(gt_c2o, rows, cols))
    )

    n = rows * cols

    if n <= 0:
        return 0.0

    adj = [[] for _ in range(n)]

    for a, b in common_edges:
        adj[a].append(b)
        adj[b].append(a)

    seen = [False] * n
    best = 1

    for i in range(n):
        if seen[i]:
            continue

        stack = [i]
        seen[i] = True
        count = 0

        while stack:
            v = stack.pop()
            count += 1

            for nb in adj[v]:
                if not seen[nb]:
                    seen[nb] = True
                    stack.append(nb)

        if count > best:
            best = count

    return float(best / n)


class PuzzleRunnerLocal:
    def __init__(self, solver, pieces_list, image_dir, output_dir, output_img_dir, y_true_csv):
        self.solver = solver
        self.pieces_list = pieces_list
        self.image_dir = image_dir
        self.output_dir = output_dir
        self.output_img_dir = output_img_dir
        self.y_true_csv = y_true_csv

        os.makedirs(self.output_dir, exist_ok=True)
        os.makedirs(self.output_img_dir, exist_ok=True)

    def run_all(self):
        results = []

        image_files = sorted([
            f for f in os.listdir(self.image_dir)
            if f.lower().endswith((".png", ".jpg", ".jpeg"))
        ])

        for idx, pieces in enumerate(tqdm(self.pieces_list, desc="Local search for images")):
            best_order, _ = self.solver.run(pieces)  # c->o, cell -> piece
            image_name = image_files[idx]

            # Lưu ảnh lắp theo c->o
            img = self.solver.assemble_image(pieces, best_order)
            img.save(os.path.join(self.output_img_dir, f"{image_name}_solved.png"))

            # Ghi trực tiếp c->o ra CSV
            results.append([image_name] + best_order[:])

        # Ghi output.csv theo c->o
        output_csv = os.path.join(self.output_dir, "output.csv")

        with open(output_csv, "w", newline="") as f:
            writer = csv.writer(f)

            header = ["image_filename"] + [
                f"piece_at_{r}_{c}"
                for r in range(self.solver.rows)
                for c in range(self.solver.cols)
            ]

            writer.writerow(header)
            writer.writerows(results)

        print(f"Saved output to {output_csv}")
        print(f"Solved images saved to {self.output_img_dir}")

        return output_csv

    def evaluate(self, output_csv):
        df_pred = pd.read_csv(output_csv)
        df_true = pd.read_csv(self.y_true_csv)

        correct_count = 0
        ppa_scores = []
        neighbor_scores = []
        lca_scores = []

        true_map = {
            row["image_filename"]: row.values[1:].astype(int)
            for _, row in df_true.iterrows()
        }

        for _, row in df_pred.iterrows():
            fname = row["image_filename"]

            if fname not in true_map:
                continue

            pred = row.values[1:].astype(int)  # output hiện tại là c->o
            gt = true_map[fname]               # gt có thể là c->o hoặc o->c
            n = len(gt)

            if set(gt) == set(range(n)):
                gt_direct = np.array(gt, dtype=int)
                gt_inv = np.array(invert_perm_o2c_to_c2o(gt), dtype=int)
                candidates = [gt_direct, gt_inv]
            else:
                candidates = [np.array(gt, dtype=int)]

            match_best = max(
                int((pred == cand).sum())
                for cand in candidates
            )

            ppa_scores.append(match_best / n if n else 0.0)

            if match_best == n:
                correct_count += 1

            neighbor_best = max(
                _neighbor_accuracy_c2o(
                    pred,
                    cand,
                    self.solver.rows,
                    self.solver.cols
                )
                for cand in candidates
            )

            lca_best = max(
                _largest_component_accuracy_c2o(
                    pred,
                    cand,
                    self.solver.rows,
                    self.solver.cols
                )
                for cand in candidates
            )

            neighbor_scores.append(neighbor_best)
            lca_scores.append(lca_best)

        total = len(df_true)

        acc = (correct_count / total) * 100 if total else 0.0
        mean_ppa = float(np.mean(ppa_scores)) if ppa_scores else 0.0
        mean_neighbor = float(np.mean(neighbor_scores)) if neighbor_scores else 0.0
        mean_lca = float(np.mean(lca_scores)) if lca_scores else 0.0

        print(f"\nTotal images: {total}")
        print(f"Correctly solved: {correct_count}/{total} ({acc:.2f}%)")
        print(f"Average Direct Accuracy (PPA): {mean_ppa:.4f}")
        print(f"Average Neighbor Accuracy: {mean_neighbor:.4f}")
        print(f"Average Largest Component Accuracy: {mean_lca:.4f}")

In [12]:
ls_solver = LocalSearchPuzzleSolver(
    rows=ROWS, cols=COLS,
    max_iters=10000,
    restarts=5,
    rng_seed=42
)

runner = PuzzleRunnerLocal(
    solver=ls_solver,
    pieces_list=all_pieces,
    image_dir=image_path,
    output_dir="output_data_private",
    output_img_dir="output_data_private/output_images",
    y_true_csv="private_test/Y_test.csv"
)

out_csv = runner.run_all()
runner.evaluate(out_csv)


Local search for images: 100%|██████████| 100/100 [00:21<00:00,  4.66it/s]

Saved output to output_data_private\output.csv
Solved images saved to output_data_private/output_images

Total images: 100
Correctly solved: 95/100 (95.00%)
Average Direct Accuracy (PPA): 0.9680
Average Neighbor Accuracy: 0.9855
Average Largest Component Accuracy: 0.9813


# Phase 2

In [13]:
# === PHASE 2: Multi-segment INSERT random refinement - FIRST IMPROVEMENT ===
import numpy as np
import os
import csv
import random
import pandas as pd
from tqdm import tqdm
from PIL import Image

# dùng lại compute_cost_matrix_mse từ phase 1


class MultiSegmentRefiner:
    """
    Multi-segment INSERT - First Improvement:
      - Vertical band: full height R, bề rộng w∈[1..C], vị trí c0.
      - Horizontal band: full width C, bề cao h∈[1..R], vị trí r0.
      - Chọn m≥2 đường cắt -> m đoạn; lấy k đoạn liên tiếp k≥1 & CHÈN vào vị trí j.

    Bug fix:
      - Chỉ loại j == i, vì reinsert đúng vị trí cũ là no-op.
      - Cho phép j == i+k vì đây vẫn là move hợp lệ.

    First Improvement:
      - Trong mỗi hàm try, gặp neighbor cải thiện đầu tiên d < 0 là return ngay.
      - Không scan hết trials để lấy best_delta nữa.

    Booster:
      - Ép thử full-width / full-height vài lần mỗi iter để bắt swap lớn
        như nửa trên ↔ nửa dưới hoặc nửa trái ↔ nửa phải.
    """

    def __init__(
        self,
        rows,
        cols,
        max_iters=2000,
        patience=200,
        trials_per_iter_v=300,
        trials_per_iter_h=300,
        booster_fw_fh_per_iter=10,
        max_segments_vertical=None,
        max_segments_horizontal=None,
        rng_seed=None
    ):
        self.rows = rows
        self.cols = cols
        self.max_iters = max_iters
        self.patience = patience
        self.trials_v = trials_per_iter_v
        self.trials_h = trials_per_iter_h
        self.booster_fw_fh = booster_fw_fh_per_iter

        self.max_seg_v = max_segments_vertical if max_segments_vertical is not None else rows
        self.max_seg_h = max_segments_horizontal if max_segments_horizontal is not None else cols

        self.rng = random.Random(rng_seed)

    # ---------- utils ----------
    def _fitness(self, chrom, H, V):
        total = 0.0
        R, C = self.rows, self.cols
        grid = np.array(chrom, dtype=int).reshape(R, C)

        for r in range(R):
            for c in range(C):
                cur = grid[r, c]

                if c < C - 1:
                    total += H[cur, grid[r, c + 1]]

                if r < R - 1:
                    total += V[cur, grid[r + 1, c]]

        return float(total)

    def _random_cuts(self, L, m):
        if L < m:
            return None

        cuts = self.rng.sample(list(range(1, L)), m - 1)
        cuts.sort()
        return tuple(cuts)

    # ---------- core random neighbors: FIRST IMPROVEMENT ----------
    def _try_vertical_insert(self, grid0, H, V, base_fit, force_full_width=False):
        """
        Thử move INSERT theo chiều dọc:
          - Cắt theo hàng trong một dải cột.
          - Gặp cải thiện đầu tiên thì return ngay.
        """
        R, C = self.rows, self.cols

        if R < 2:
            return None, 0.0

        mmax = max(2, min(R, self.max_seg_v))
        trials = self.booster_fw_fh if force_full_width else self.trials_v

        for _ in range(trials):
            if force_full_width:
                w, c0 = C, 0

                # Ưu tiên m nhỏ để bắt swap lớn.
                if R >= 3:
                    m = self.rng.choice([2, 3])
                else:
                    m = 2
            else:
                w = self.rng.randint(1, C)
                c0 = self.rng.randint(0, C - w)
                m = self.rng.randint(2, mmax)

            cuts = self._random_cuts(R, m)
            if cuts is None:
                continue

            # Chia theo HÀNG trong dải cột [c0..c0+w-1]
            segments = []
            prev = 0

            for cut in cuts:
                segments.append(grid0[prev:cut, c0:c0 + w].copy())
                prev = cut

            segments.append(grid0[prev:R, c0:c0 + w].copy())

            k = self.rng.randint(1, m - 1)
            i = self.rng.randint(0, m - k)

            # Bug fix:
            # Chỉ loại j == i vì đó là no-op.
            # j == i+k vẫn hợp lệ.
            possible_js = [jj for jj in range(0, m - k + 1) if jj != i]

            if not possible_js:
                continue

            j = self.rng.choice(possible_js)

            g = grid0.copy()

            block = segments[i:i + k]
            rest = segments[:i] + segments[i + k:]
            new_order = rest[:j] + block + rest[j:]

            r_ptr = 0
            for seg in new_order:
                rs = seg.shape[0]
                g[r_ptr:r_ptr + rs, c0:c0 + w] = seg
                r_ptr += rs

            cand = g.reshape(-1).tolist()
            d = self._fitness(cand, H, V) - base_fit

            # FIRST IMPROVEMENT
            if d < 0.0:
                return cand, d

        return None, 0.0

    def _try_horizontal_insert(self, grid0, H, V, base_fit, force_full_height=False):
        """
        Thử move INSERT theo chiều ngang:
          - Cắt theo cột trong một dải hàng.
          - Gặp cải thiện đầu tiên thì return ngay.
        """
        R, C = self.rows, self.cols

        if C < 2:
            return None, 0.0

        mmax = max(2, min(C, self.max_seg_h))
        trials = self.booster_fw_fh if force_full_height else self.trials_h

        for _ in range(trials):
            if force_full_height:
                h, r0 = R, 0

                # Ưu tiên m nhỏ để bắt swap lớn.
                if C >= 3:
                    m = self.rng.choice([2, 3])
                else:
                    m = 2
            else:
                h = self.rng.randint(1, R)
                r0 = self.rng.randint(0, R - h)
                m = self.rng.randint(2, mmax)

            cuts = self._random_cuts(C, m)
            if cuts is None:
                continue

            # Chia theo CỘT trong dải hàng [r0..r0+h-1]
            segments = []
            prev = 0

            for cut in cuts:
                segments.append(grid0[r0:r0 + h, prev:cut].copy())
                prev = cut

            segments.append(grid0[r0:r0 + h, prev:C].copy())

            k = self.rng.randint(1, m - 1)
            i = self.rng.randint(0, m - k)

            # Bug fix tương tự:
            # Chỉ loại j == i, cho phép j == i+k.
            possible_js = [jj for jj in range(0, m - k + 1) if jj != i]

            if not possible_js:
                continue

            j = self.rng.choice(possible_js)

            g = grid0.copy()

            block = segments[i:i + k]
            rest = segments[:i] + segments[i + k:]
            new_order = rest[:j] + block + rest[j:]

            c_ptr = 0
            for seg in new_order:
                cs = seg.shape[1]
                g[r0:r0 + h, c_ptr:c_ptr + cs] = seg
                c_ptr += cs

            cand = g.reshape(-1).tolist()
            d = self._fitness(cand, H, V) - base_fit

            # FIRST IMPROVEMENT
            if d < 0.0:
                return cand, d

        return None, 0.0

    def refine(self, pieces, init_chrom):
        H, V = compute_cost_matrix_mse(pieces)

        best = init_chrom[:]
        best_fit = self._fitness(best, H, V)

        no_improve = 0
        history = [(0, best_fit)]

        for it in range(1, self.max_iters + 1):
            grid0 = np.array(best, dtype=int).reshape(self.rows, self.cols)

            chosen = None
            delta = 0.0

            # FIRST IMPROVEMENT theo thứ tự ưu tiên:
            # 1. Booster full-width: tốt để bắt đổi hàng / block hàng lớn.
            chosen, delta = self._try_vertical_insert(
                grid0,
                H,
                V,
                best_fit,
                force_full_width=True
            )

            # 2. Booster full-height: tốt để bắt đổi cột / block cột lớn.
            if chosen is None:
                chosen, delta = self._try_horizontal_insert(
                    grid0,
                    H,
                    V,
                    best_fit,
                    force_full_height=True
                )

            # 3. Random vertical band.
            if chosen is None:
                chosen, delta = self._try_vertical_insert(
                    grid0,
                    H,
                    V,
                    best_fit,
                    force_full_width=False
                )

            # 4. Random horizontal band.
            if chosen is None:
                chosen, delta = self._try_horizontal_insert(
                    grid0,
                    H,
                    V,
                    best_fit,
                    force_full_height=False
                )

            if chosen is not None and delta < 0.0:
                best = chosen
                best_fit += delta
                history.append((it, best_fit))
                no_improve = 0
            else:
                no_improve += 1

                if no_improve >= self.patience:
                    break

        return best, best_fit, history


class Phase2Runner:
    """
    Đọc output phase 1 dạng c->o, refine từng ảnh bằng MultiSegmentRefiner,
    ghi output_phase2.csv dạng c->o và ảnh *_refined.png.
    """

    def __init__(
        self,
        refiner,
        pieces_list,
        image_dir,
        output_dir,
        output_img_dir,
        y_true_csv,
        assembler=None
    ):
        self.refiner = refiner
        self.pieces_list = pieces_list
        self.image_dir = image_dir
        self.output_dir = output_dir
        self.output_img_dir = output_img_dir
        self.y_true_csv = y_true_csv
        self.assembler = assembler

        os.makedirs(self.output_dir, exist_ok=True)
        os.makedirs(self.output_img_dir, exist_ok=True)

    def _assemble_image(self, pieces, order):
        if self.assembler is not None:
            return self.assembler(pieces, order)

        R, C = self.refiner.rows, self.refiner.cols
        grid = np.array(order, dtype=int).reshape(R, C)

        w, h = pieces[0].size
        out = Image.new("RGB", (w * C, h * R))

        for r in range(R):
            for c in range(C):
                out.paste(pieces[grid[r, c]], (c * w, r * h))

        return out

    def refine_all(self, input_csv_phase1, output_csv_phase2=None):
        if output_csv_phase2 is None:
            output_csv_phase2 = os.path.join(self.output_dir, "output_phase2.csv")

        image_files = sorted([
            f for f in os.listdir(self.image_dir)
            if f.lower().endswith((".png", ".jpg", ".jpeg"))
        ])

        df_p1 = pd.read_csv(input_csv_phase1)

        pred_map = {
            row["image_filename"]: row.values[1:].astype(int).tolist()
            for _, row in df_p1.iterrows()
        }

        results = []

        for idx, pieces in enumerate(tqdm(self.pieces_list, desc="Phase 2 refining")):
            fname = image_files[idx]

            if fname not in pred_map:
                continue

            init_chrom = pred_map[fname]

            best, best_fit, _ = self.refiner.refine(pieces, init_chrom)

            img = self._assemble_image(pieces, best)
            img.save(os.path.join(self.output_img_dir, f"{fname}_refined.png"))

            results.append([fname] + best)

        with open(output_csv_phase2, "w", newline="") as f:
            writer = csv.writer(f)

            header = ["image_filename"] + [
                f"piece_at_{r}_{c}"
                for r in range(self.refiner.rows)
                for c in range(self.refiner.cols)
            ]

            writer.writerow(header)
            writer.writerows(results)

        print(f"Saved phase-2 output to {output_csv_phase2}")
        print(f"Refined images saved to {self.output_img_dir}")

        return output_csv_phase2

In [14]:
# === Sau khi bạn đã chạy Phase 1 như trong code hiện tại ===
# out_csv = runner.run_all()
# runner.evaluate(out_csv)

# Tạo refiner (random sampling). Tăng trials/iters nếu muốn tìm kỹ hơn.
refiner = MultiSegmentRefiner(
    rows=ROWS, cols=COLS,
    max_iters=3000,              # số vòng refine tối đa
    patience=300,                # dừng sớm nếu không cải thiện
    trials_per_iter_v=600,       # lân cận vertical / iter
    trials_per_iter_h=600,       # lân cận horizontal / iter
    max_segments_vertical=ROWS,  # m tối đa theo chiều dọc (thường = ROWS)
    max_segments_horizontal=COLS,# m tối đa theo chiều ngang (thường = COLS)
    rng_seed=123                 # random seed cho phase 2
)

# Dùng assemble của solver phase 1 để vẽ ảnh
phase2 = Phase2Runner(
    refiner=refiner,
    pieces_list=all_pieces,
    image_dir=image_path,
    output_dir="output_data_private",                 # CSV refined vào đây
    output_img_dir="output_data_private/refined_images",     # Ảnh refined vào đây
    y_true_csv="private_test/Y_test.csv",
    assembler=ls_solver.assemble_image               # tái sử dụng assembler của solver phase 1
)

out_csv_phase2 = phase2.refine_all(
    input_csv_phase1=os.path.join("output_data_private", "output.csv"),
    output_csv_phase2=os.path.join("output_data_private", "output_phase2.csv")
)

# Đánh giá phase 2 (tái dùng evaluator hiện có của bạn)
runner.evaluate(out_csv_phase2)


Phase 2 refining: 100%|██████████| 100/100 [11:22<00:00,  6.83s/it]

Saved phase-2 output to output_data_private\output_phase2.csv
Refined images saved to output_data_private/refined_images

Total images: 100
Correctly solved: 97/100 (97.00%)
Average Direct Accuracy (PPA): 0.9800
Average Neighbor Accuracy: 0.9909
Average Largest Component Accuracy: 0.9880
